# 05 — MCMC: Metropolis-Hastings
**References:** Brooks, Gelman, Jones & Meng (2011) Ch. 1–3 · Gelman et al. BDA3 Ch. 11 · Tierney (1994)

## Narrative thread
```
Markov chains -> Detailed balance -> MH algorithm -> Proposal tuning -> Convergence diagnostics -> Multiple chains
```

## 1. Markov chains and stationarity

A **Markov chain** is a sequence $\theta^{(1)}, \theta^{(2)}, \ldots$ where:
$$p(\theta^{(t+1)} \mid \theta^{(t)}, \theta^{(t-1)}, \ldots) = p(\theta^{(t+1)} \mid \theta^{(t)}) \quad \text{(Markov property)}$$

A distribution $\pi$ is **stationary** for the chain if $\theta^{(t)} \sim \pi \Rightarrow \theta^{(t+1)} \sim \pi$.

**Detailed balance** (sufficient for stationarity):
$$\pi(\theta)\, T(\theta' \mid \theta) = \pi(\theta')\, T(\theta \mid \theta') \quad \forall\,\theta,\theta'$$

A chain that is **irreducible** (can reach any state) and **aperiodic** converges to its unique stationary distribution regardless of the starting point.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Metropolis-Hastings: generic implementation ───────────────────────────
def metropolis_hastings(log_target, proposal_sd, n_iter, theta_init):
    """Random-walk Metropolis-Hastings."""
    chain      = np.zeros(n_iter)
    chain[0]   = theta_init
    n_accept   = 0
    for t in range(1, n_iter):
        current  = chain[t-1]
        proposed = current + np.random.normal(0, proposal_sd)
        log_r    = log_target(proposed) - log_target(current)
        if np.log(np.random.uniform()) < log_r:
            chain[t]  = proposed
            n_accept += 1
        else:
            chain[t] = current
    return chain, n_accept / (n_iter - 1)

# Target: mixture of two Gaussians
log_target = lambda t: np.log(0.4 * stats.norm.pdf(t, -2, 0.8) + 0.6 * stats.norm.pdf(t, 3, 1.2))

n_iter     = 20_000
burn_in    = 2_000
theta_grid = np.linspace(-6, 8, 400)
true_pdf   = 0.4*stats.norm.pdf(theta_grid, -2, 0.8) + 0.6*stats.norm.pdf(theta_grid, 3, 1.2)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, proposal_sd in enumerate([0.1, 1.0, 10.0]):
    chain, accept_rate = metropolis_hastings(log_target, proposal_sd, n_iter, theta_init=0.0)
    post_chain = chain[burn_in:]

    axes[0, col].plot(chain[:500], lw=0.8, color='#4361ee', alpha=0.8)
    axes[0, col].axvline(burn_in if burn_in<500 else 500, color='#f72585', lw=1.5, linestyle='--')
    axes[0, col].set_title(f'proposal_sd={proposal_sd}\nAccept rate={accept_rate:.2f}')
    axes[0, col].set_xlabel('Iteration'); axes[0, col].set_ylabel('$\\theta$')

    axes[1, col].hist(post_chain, bins=80, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
    axes[1, col].plot(theta_grid, true_pdf, 'r-', lw=2.5, label='True target')
    axes[1, col].set_xlabel('$\\theta$'); axes[1, col].set_ylabel('Density')
    axes[1, col].legend(fontsize=8)
    label = {0.1: 'Too small: slow exploration', 1.0: 'Good: ~23% accept optimal', 10.0: 'Too large: high rejection'}[proposal_sd]
    axes[1, col].set_title(f'{label}')

plt.suptitle('Metropolis-Hastings: Effect of Proposal Scale on Mixing\n'
             'Optimal accept rate ≈ 23% for 1D; 44% for scalar MH',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The MH acceptance step

Given current state $\theta$ and proposal $\theta' \sim q(\theta' \mid \theta)$, accept with probability:

$$\alpha(\theta, \theta') = \min\!\left(1,\, \frac{\tilde p(\theta')}{\tilde p(\theta)} \cdot \frac{q(\theta \mid \theta')}{q(\theta' \mid \theta)}\right)$$

where $\tilde p = p(\mathbf{x}\mid\theta)\,p(\theta)$ is the unnormalized posterior. The normalizing constant cancels!

**Special cases:**
- **Random walk MH:** $q(\theta'\mid\theta) = \mathcal{N}(\theta, \sigma^2)$ — symmetric, so $q$ terms cancel
- **Independence sampler:** $q(\theta'\mid\theta) = q(\theta')$ — proposal doesn't depend on current state
- **Metropolis (original):** Symmetric proposal — $q(\theta'\mid\theta) = q(\theta\mid\theta')$

**Optimal acceptance rate:** Roberts, Gelman & Gilks (1997) showed that for a $d$-dimensional random walk MH targeting a product-normal:
$$\text{optimal acceptance rate} \approx 0.234 \quad (d\to\infty), \quad 0.44 \quad (d=1)$$

## 3. Convergence diagnostics

MCMC requires diagnostics to verify the chain has converged to the target.

**$\hat R$ (Gelman-Rubin):** Run $M$ chains. The **potential scale reduction factor** is:
$$\hat R = \sqrt{\frac{\hat{\text{Var}}^+(\theta \mid y)}{W}}$$
where $W$ is the within-chain variance and $\hat{\text{Var}}^+$ is the mixture variance. $\hat R \approx 1$ when chains agree. $\hat R > 1.1$ signals non-convergence.

**Effective sample size (ESS):**
$$n_{\text{eff}} = \frac{n}{1 + 2\sum_{t=1}^\infty \rho_t}$$
where $\rho_t$ is the lag-$t$ autocorrelation. High autocorrelation → low ESS → more iterations needed.

**Trace plots:** Visual check — should look like a "hairy caterpillar" after burn-in.

In [ ]:
# ── Convergence diagnostics: R-hat and ESS ───────────────────────────────
def gelman_rubin(chains):
    """Compute R-hat for a list of chains (each already post-warmup)."""
    M  = len(chains)
    N  = len(chains[0])
    chain_means = np.array([c.mean() for c in chains])
    chain_vars  = np.array([c.var(ddof=1) for c in chains])
    grand_mean  = chain_means.mean()
    B = N / (M - 1) * np.sum((chain_means - grand_mean)**2)  # between-chain variance * N
    W = chain_vars.mean()                                      # within-chain variance
    var_plus = (N-1)/N * W + B/N
    return np.sqrt(var_plus / W)

def effective_ss(chain):
    """Estimate effective sample size via autocorrelation."""
    n    = len(chain)
    acf  = np.array([np.corrcoef(chain[:-k], chain[k:])[0,1] if k>0 else 1.0 for k in range(min(n//2, 200))])
    # Sum until acf drops below 0.05 or alternates
    total_ac = 1 + 2 * sum(r for r in acf[1:] if r > 0.05)
    return n / total_ac

# Run 4 chains from different starting points
log_target_gauss = lambda t: stats.norm.logpdf(t, 5, 2)
n_iter_diag = 5000; warmup = 1000

starts  = [-10, -5, 15, 20]  # overdispersed starts
chains  = []
for s in starts:
    c, _ = metropolis_hastings(log_target_gauss, proposal_sd=2.0, n_iter=n_iter_diag, theta_init=s)
    chains.append(c)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors4 = ['#4361ee', '#f72585', '#06d6a0', '#ff9f1c']

# Trace plots — full chain
ax = axes[0, 0]
for c, col, s in zip(chains, colors4, starts):
    ax.plot(c, lw=0.5, color=col, alpha=0.8, label=f'Start={s}')
ax.axvline(warmup, color='k', lw=2, linestyle='--', label='End of warmup')
ax.set_xlabel('Iteration'); ax.set_ylabel('$\\theta$')
ax.set_title('Trace plots: 4 chains\nAll should converge and mix well')
ax.legend(fontsize=7)

# R-hat over time
ax2 = axes[0, 1]
t_vals = np.arange(100, n_iter_diag, 50)
r_hats = [gelman_rubin([c[:t] for c in chains]) for t in t_vals]
ax2.plot(t_vals, r_hats, color='#4361ee', lw=2)
ax2.axhline(1.1, color='#f72585', lw=2, linestyle='--', label='$\\hat R = 1.1$ threshold')
ax2.axhline(1.0, color='#06d6a0', lw=1.5, linestyle=':', label='$\\hat R = 1.0$ (ideal)')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('$\\hat R$')
ax2.set_title('$\\hat R$ over time\nDrops to ~1 after warmup')
ax2.legend(fontsize=8)

# Post-warmup chains + marginal
post_chains = [c[warmup:] for c in chains]
all_post    = np.concatenate(post_chains)
ax3 = axes[1, 0]
for c, col, s in zip(post_chains, colors4, starts):
    ax3.plot(c[:500], lw=0.7, color=col, alpha=0.7)
ax3.set_xlabel('Post-warmup iteration'); ax3.set_ylabel('$\\theta$')
ax3.set_title(f'Post-warmup traces\n$\\hat R = {gelman_rubin(post_chains):.4f}$')

# ESS per chain
ax4 = axes[1, 1]
ess_vals = [effective_ss(c) for c in post_chains]
ax4.bar([f'Chain {i+1}\n(start={s})' for i,s in enumerate(starts)], ess_vals,
        color=colors4, alpha=0.8)
for i, e in enumerate(ess_vals):
    ax4.text(i, e + 10, f'{e:.0f}', ha='center', fontweight='bold')
ax4.set_ylabel('Effective sample size')
ax4.set_title(f'ESS per chain (total post-warmup: {n_iter_diag-warmup})\nESS/total = mixing efficiency')

plt.suptitle('MCMC Convergence Diagnostics: $\\hat R$ and ESS', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nFinal R-hat (post-warmup): {gelman_rubin(post_chains):.5f}  (want < 1.01)')
print(f'Total post-warmup samples: {len(all_post)}')
print(f'Total ESS: {effective_ss(all_post):.0f}')

## 4. Key takeaways

| Concept | Statement |
|---|---|
| MH acceptance | $\min(1, \tilde p(\theta')/\tilde p(\theta) \cdot q(\theta)/q(\theta'))$ |
| Normalizing constant | Cancels in acceptance ratio — only unnormalized posterior needed |
| Optimal accept rate | ~23% for high-d random walk MH |
| $\hat R < 1.01$ | Convergence criterion (Vehtari et al. 2021) |
| ESS | Effective samples after accounting for autocorrelation |
| Warmup | Early iterations discarded — chain exploring, not yet stationary |

**Next:** notebook 06 — Gibbs sampling and Hamiltonian Monte Carlo: smarter MCMC algorithms.